# OrbitWarsSimulator — correctness tests
Compare `OrbitWarsSimulator` from `27-Board_new_env.py` against the real
`kaggle_environments` interpreter step-by-step.

In [19]:
import importlib, sys, math, kaggle_environments as ke

# Hot-reload the module so edits take effect without restarting the kernel
spec = importlib.util.spec_from_file_location(
    "board27",
    r"C:\Users\trant\Documents\Programmation\Orbit Wars\27-Board_new_env.py"
)
board27 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(board27)
OrbitWarsSimulator = board27.OrbitWarsSimulator
print("Loaded OrbitWarsSimulator")

Loaded OrbitWarsSimulator


## Helper — extract obs as plain dict

In [20]:
def obs_to_dict(obs):
    """Convert a kaggle Struct observation to a plain dict."""
    return {
        "angular_velocity": obs.angular_velocity,
        # Real ke env uses obs.step as the rotation step index; must be forwarded.
        "step": obs.step,
        "comet_planet_ids": list(obs.comet_planet_ids),
        "planets": [list(p) for p in obs.planets],
        # Game-start positions needed for the absolute rotation formula.
        "initial_planets": [list(p) for p in obs.initial_planets],
        "fleets": [list(f) for f in obs.fleets],
        "comets": [
            {
                "planet_ids": list(g.planet_ids),
                "paths": [[list(pos) for pos in path] for path in g.paths],
                "path_index": g.path_index,
            }
            for g in (getattr(obs, "comets", None) or [])
        ],
    }

def planet_snapshot(planets):
    """List of (id, owner, x, y, ships) tuples for quick display."""
    return [(p[0], p[1], round(p[2],3), round(p[3],3), p[5]) for p in planets]

print("helpers ready")

helpers ready


## Test 1 — Production
Non-orbiting planets with no fleets. Ships should increase by `production` each step.

In [21]:
# planet format: [id, owner, x, y, radius, ships, production]
obs_prod = {
    "angular_velocity": 0.0,
    "comet_planet_ids": [],
    "planets": [
        [0, 1,  20.0, 50.0, 5.0, 10, 3],  # owned by player 1, prod=3
        [1, -1, 80.0, 50.0, 5.0, 5,  2],  # neutral, no production
    ],
    "fleets": [],
    "comets": [],
}

sim = OrbitWarsSimulator(obs_prod)
snaps = sim.run(3)
print("After step 1:", [(p['id'], p['ships']) for p in snaps[0]])
print("After step 2:", [(p['id'], p['ships']) for p in snaps[1]])
print("After step 3:", [(p['id'], p['ships']) for p in snaps[2]])

# Planet 0: 10 + 3 = 13, +3 = 16, +3 = 19
# Planet 1: neutral → no production (owner == -1)
assert snaps[0][0]['ships'] == 13, f"Expected 13, got {snaps[0][0]['ships']}"
assert snaps[2][0]['ships'] == 19, f"Expected 19, got {snaps[2][0]['ships']}"
assert snaps[2][1]['ships'] == 5,  f"Neutral should stay 5, got {snaps[2][1]['ships']}"
print("PASS")

After step 1: [(0, 13), (1, 5)]
After step 2: [(0, 16), (1, 5)]
After step 3: [(0, 19), (1, 5)]
PASS


## Test 2 — Fleet movement and arrival
Fire a fleet from planet 0 toward planet 1 and confirm it arrives and resolves combat.

In [22]:
# Two planets at (20,50) and (80,50), no rotation.
# Fleet: [id, owner, x, y, angle, ?, ships]
# A fleet with 1 ship has speed 1.0; distance surface-to-surface = 60 - 5 - 5 = 50 steps.
obs_fleet = {
    "angular_velocity": 0.0,
    "comet_planet_ids": [],
    "planets": [
        [0, 1, 20.0, 50.0, 5.0, 10, 3],
        [1, 0, 80.0, 50.0, 5.0,  8, 2],
    ],
    "fleets": [
        # Spawns at planet-0 surface: x = 20 + cos(0)*5 = 25
        [99, 1, 25.0, 50.0, 0.0, 0, 1],  # 1 ship, angle=0 (east)
    ],
    "comets": [],
}

sim = OrbitWarsSimulator(obs_fleet)

# The fleet starts at x=25 and needs to reach x=75 (planet surface), travel=50, speed=1 → 50 steps
snaps = sim.run(55)

# Check fleet disappears (absorbed by planet) within 55 steps
alive_at_end = len(sim.fleets)
print(f"Fleet count after 55 steps: {alive_at_end}  (should be 0)")

# Combat: 1 attacker vs 8 defenders → defenders win with 8-1=7 ships, planet stays owner 0
planet1 = next(p for p in snaps[-1] if p['id'] == 1)
print(f"Planet 1 owner={planet1['owner']}, ships={planet1['ships']}")
assert alive_at_end == 0, "Fleet should be gone"
assert planet1['owner'] == 0, f"Planet 1 should still belong to player 0, got {planet1['owner']}"
print("PASS")

Fleet count after 55 steps: 0  (should be 0)
Planet 1 owner=0, ships=118
PASS


## Test 3 — Orbit correctness vs real ke env (step-by-step)
Start a fresh game, then at each game step compare what our simulator predicts for step+1
against what the real env actually produces.

In [23]:
COMPARE_STEPS = 10  # how many steps to validate
THRESHOLD = 0.01    # max allowed position error (float rounding)

env = ke.make("orbit_wars", debug=True)
env.reset()

errors = []

for step in range(COMPARE_STEPS):
    obs = env.state[0].observation
    d = obs_to_dict(obs)

    # Run our simulator for 1 step
    sim_snap = OrbitWarsSimulator(d).step()  # returns list of planet dicts

    # Advance the real env (no moves)
    env.step([[], []])
    real_obs = env.state[0].observation

    # Compare planet positions and ship counts
    real_by_id = {p[0]: p for p in real_obs.planets}
    for sp in sim_snap:
        pid = sp['id']
        if pid not in real_by_id:
            continue  # comet expired — skip
        rp = real_by_id[pid]
        dx = abs(sp['x'] - rp[2])
        dy = abs(sp['y'] - rp[3])
        ds = sp['ships'] - rp[5]
        if dx > THRESHOLD or dy > THRESHOLD:
            errors.append(f"step={step+1} pid={pid} pos sim=({sp['x']:.3f},{sp['y']:.3f}) real=({rp[2]:.3f},{rp[3]:.3f}) Δ=({dx:.4f},{dy:.4f})")
        if ds != 0:
            errors.append(f"step={step+1} pid={pid} ships sim={sp['ships']} real={rp[5]} Δ={ds}")

if errors:
    print(f"FAIL — {len(errors)} discrepancies:")
    for e in errors:
        print(" ", e)
else:
    print(f"PASS — {COMPARE_STEPS} steps, all positions/ships match within {THRESHOLD}")

PASS — 10 steps, all positions/ships match within 0.01


## Test 4 — Orbit correctness mid-game (step 50)
Advance to step 50 with no moves, then compare one more simulator step against the real env.
This catches the step-offset bug diagnosed in `23-next10.ipynb`.

In [24]:
ADVANCE_TO = 50
THRESHOLD  = 0.01

env2 = ke.make("orbit_wars", debug=True)
env2.reset()
for _ in range(ADVANCE_TO):
    env2.step([[], []])

obs_mid = env2.state[0].observation
av      = obs_mid.angular_velocity
game_step = obs_mid.step
print(f"Game at step {game_step}, ω={av:.4f}")

# Print orbiting planets
orb_planets = [p for p in obs_mid.planets if math.hypot(p[2]-50, p[3]-50) + p[4] < 50]
print(f"Orbiting planets: {[p[0] for p in orb_planets]}")

# Simulator prediction (uses current planets as reference, sim_step starts at 0)
d_mid   = obs_to_dict(obs_mid)
sim_mid = OrbitWarsSimulator(d_mid).step()

# Real env next step
env2.step([[], []])
real_mid = env2.state[0].observation
real_by_id_mid = {p[0]: p for p in real_mid.planets}

errors_mid = []
for sp in sim_mid:
    pid = sp['id']
    if pid not in real_by_id_mid:
        continue
    rp = real_by_id_mid[pid]
    dx = abs(sp['x'] - rp[2])
    dy = abs(sp['y'] - rp[3])
    ds = sp['ships'] - rp[5]
    if dx > THRESHOLD or dy > THRESHOLD:
        is_orb = pid in [p[0] for p in orb_planets]
        errors_mid.append(f"  pid={pid} {'(orbiting)' if is_orb else ''} pos sim=({sp['x']:.3f},{sp['y']:.3f}) real=({rp[2]:.3f},{rp[3]:.3f}) Δ=({dx:.4f},{dy:.4f})")
    if ds != 0:
        errors_mid.append(f"  pid={pid} ships sim={sp['ships']} real={rp[5]} Δ={ds}")

if errors_mid:
    print(f"FAIL — {len(errors_mid)} discrepancies at step {game_step}→{game_step+1}:")
    for e in errors_mid:
        print(e)
else:
    print(f"PASS — step {game_step}→{game_step+1} all positions match")

Game at step 50, ω=0.0267
Orbiting planets: [12, 13, 14, 15, 20, 21, 22, 23, 28, 29, 30, 31, 32, 33, 34, 35]
PASS — step 50→51 all positions match


## Test 5 — Combat resolution
Multiple fleets arrive at the same planet in the same step.
Verify winner and survivor count match the expected formula.

In [25]:
# Planet 0: neutral at (50,50), radius 5, 0 ships.
# Fleet A (player 1): 10 ships.  Speed=1.  Starts at x=44.5 (just inside the surface
#   approach zone). After 1 step: old=(44.5,50) new=(45.5,50).
#   pt_seg_dist((50,50), (44.5,50),(45.5,50)) = dist to closest end (45.5,50) = 4.5 < 5. HIT.
# Fleet B (player 2): 6 ships.  Starts at x=55.5 moving west.
#   After 1 step: old=(55.5,50) new=(54.5,50). dist from (50,50) to closest end = 4.5 < 5. HIT.
# Expected: player 1 wins 10 vs 6 → 4 survivors → planet becomes owner=1, ships=4.
obs_combat = {
    "angular_velocity": 0.0,
    "step": 0,
    "comet_planet_ids": [],
    "planets": [
        [0, -1, 50.0, 50.0, 5.0, 0, 0],
    ],
    "fleets": [
        [10, 1, 44.5, 50.0, 0.0,     0, 10],   # moving east
        [11, 2, 55.5, 50.0, math.pi, 0,  6],   # moving west
    ],
    "comets": [],
}

sim = OrbitWarsSimulator(obs_combat)
snap1 = sim.step()
print("After step 1:", snap1)

planet0 = snap1[0]
print(f"owner={planet0['owner']}, ships={planet0['ships']}")
assert planet0['owner'] == 1, f"Expected owner=1 (player 1 wins), got {planet0['owner']}"
assert planet0['ships'] == 4, f"Expected 4 survivors, got {planet0['ships']}"
print("PASS")

After step 1: [{'id': 0, 'owner': 1, 'x': 50.0, 'y': 50.0, 'radius': 5.0, 'ships': 4, 'production': 0}]
owner=1, ships=4
PASS


## Test 6 — Sun destruction
A fleet fired toward the sun should disappear without reaching any planet.

In [26]:
# Planet at (20, 50); fleet fired east toward the sun (center=50,50, radius=10).
# Fleet at (25, 50) moving east at speed 1 → hits sun in ~15 steps.
obs_sun = {
    "angular_velocity": 0.0,
    "comet_planet_ids": [],
    "planets": [
        [0, 1, 20.0, 50.0, 5.0, 10, 2],
        [1, 0, 80.0, 50.0, 5.0, 10, 2],
    ],
    "fleets": [
        [99, 1, 26.0, 50.0, 0.0, 0, 1],  # aimed east, will pass through sun
    ],
    "comets": [],
}

sim = OrbitWarsSimulator(obs_sun)
snaps = sim.run(50)

fleet_gone = len(sim.fleets) == 0
# Also verify planet 1 (at 80,50) was NOT captured (fleet should have died in sun)
planet1 = next(p for p in snaps[-1] if p['id'] == 1)
print(f"Fleet gone after 50 steps: {fleet_gone}")
print(f"Planet 1 still owner=0: {planet1['owner'] == 0}")
assert fleet_gone, "Fleet aimed at sun should be destroyed"
assert planet1['owner'] == 0, "Planet 1 should not have been captured"
print("PASS")

Fleet gone after 50 steps: True
Planet 1 still owner=0: True
PASS


## Test 7 — Multi-step sequential comparison against real env
Play 30 steps feeding our simulator from the *current* obs each time, accumulating errors.

In [27]:
STEPS      = 30
THRESHOLD  = 0.01
all_errors = []

env3 = ke.make("orbit_wars", debug=True)
env3.reset()

for step in range(STEPS):
    obs_now = env3.state[0].observation
    d_now   = obs_to_dict(obs_now)
    sim_snap = OrbitWarsSimulator(d_now).step()

    env3.step([[], []])
    real_now   = env3.state[0].observation
    real_by_id = {p[0]: p for p in real_now.planets}

    for sp in sim_snap:
        pid = sp['id']
        if pid not in real_by_id:
            continue
        rp = real_by_id[pid]
        dx = abs(sp['x'] - rp[2])
        dy = abs(sp['y'] - rp[3])
        ds = sp['ships'] - rp[5]
        if dx > THRESHOLD or dy > THRESHOLD:
            all_errors.append(f"step={step+1} pid={pid} Δpos=({dx:.4f},{dy:.4f})")
        if ds != 0:
            all_errors.append(f"step={step+1} pid={pid} Δships={ds}")

if all_errors:
    print(f"FAIL — {len(all_errors)} discrepancies over {STEPS} steps:")
    for e in all_errors[:20]:  # cap output
        print(" ", e)
    if len(all_errors) > 20:
        print(f"  ... and {len(all_errors)-20} more")
else:
    print(f"PASS — {STEPS} steps, zero discrepancies")

PASS — 30 steps, zero discrepancies


# Manual tests

In [29]:
env = ke.make("orbit_wars", debug=True)
env.reset()
print(env.render(mode="ansi"))

Step 0
Planets:
  ID: 0, Owner: -1, Pos: (93.0, 74.2), R: 2.1, Ships: 6, Prod: 3
  ID: 1, Owner: -1, Pos: (25.8, 93.0), R: 2.1, Ships: 6, Prod: 3
  ID: 2, Owner: -1, Pos: (74.2, 7.0), R: 2.1, Ships: 6, Prod: 3
  ID: 3, Owner: -1, Pos: (7.0, 25.8), R: 2.1, Ships: 6, Prod: 3
  ID: 4, Owner: -1, Pos: (90.2, 85.0), R: 1.0, Ships: 72, Prod: 1
  ID: 5, Owner: -1, Pos: (15.0, 90.2), R: 1.0, Ships: 72, Prod: 1
  ID: 6, Owner: -1, Pos: (85.0, 9.8), R: 1.0, Ships: 72, Prod: 1
  ID: 7, Owner: -1, Pos: (9.8, 15.0), R: 1.0, Ships: 72, Prod: 1
  ID: 8, Owner: 0, Pos: (57.6, 98.5), R: 1.0, Ships: 10, Prod: 1
  ID: 9, Owner: -1, Pos: (1.5, 57.6), R: 1.0, Ships: 83, Prod: 1
  ID: 10, Owner: -1, Pos: (98.5, 42.4), R: 1.0, Ships: 83, Prod: 1
  ID: 11, Owner: 1, Pos: (42.4, 1.5), R: 1.0, Ships: 10, Prod: 1
  ID: 12, Owner: -1, Pos: (80.7, 93.1), R: 1.0, Ships: 17, Prod: 1
  ID: 13, Owner: -1, Pos: (6.9, 80.7), R: 1.0, Ships: 17, Prod: 1
  ID: 14, Owner: -1, Pos: (93.1, 19.3), R: 1.0, Ships: 17, Prod: 1
  

In [ ]:
obs = env.state[0].observation
d = obs_to_dict(obs)

# Run our simulator for 1 step
sim_snap = OrbitWarsSimulator(d)
sim_snap.render()